In [ ]:
# Setup: find the course materials and load our helper code. Run this cell first.
import subprocess
import sys
from pathlib import Path

# Colab has a module called google.colab; a laptop does not.
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # On Colab: download the course materials once.
    TASK_DIR = Path("/content/intern-training/tasks/esm2-enzyme-transfer")
    if not TASK_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "esm2-v1",
             "https://github.com/lagillenwater/intern-training", "/content/intern-training"],
            check=True,
        )
    # Install the exact package versions this course was tested with.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", 'accelerate==1.16.0', 'transformers==5.23.0', 'umap-learn==0.6.0'],
        check=True,
    )
else:
    # On a laptop: walk up from the current folder until we find the folder
    # that holds both helpers/ and checks.py.
    TASK_DIR = None
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "helpers").is_dir() and (folder / "checks.py").exists():
            TASK_DIR = folder
            break
    if TASK_DIR is None:
        raise RuntimeError("Could not find the course materials (a folder with helpers/ and checks.py).")

# Make our helper code importable, then load it.
sys.path.insert(0, str(TASK_DIR))
from helpers import *
from checks import *

print(f"Setup complete. Materials: {TASK_DIR}")


# 02 Build the baseline

**What this step is for:** build the model that fine-tuning has to beat. You use ESM-2 exactly as it was downloaded, without changing it, to turn every protein into a list of numbers. Then you train a simple classifier on the human training proteins, score it on the human test proteins, and save its predictions for every test protein so notebook 04 can compare it with the fine-tuned model.

**Time:** about 1.25 hours. Right after this notebook you have check-in 1 with your mentor.

**Reads:** `data/enzymes.tsv`. **Writes:** `embeddings_base.npz` and `predictions_baseline.tsv` in your run folder on Google Drive.

**Before you start:** use a GPU runtime for this notebook (in Colab: Runtime > Change runtime type > T4 GPU; the README shows how). Then run the Setup cell at the top and work top to bottom. If a cell below a `TODO` shows an error, first make sure that `TODO` passed its check.

In [ ]:
# Configuration for this run (the same in every notebook).
RUN_NAME = "esm2_8M"                      # a label for this run: the small ESM-2 model, with 8 million parameters
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"  # the model's name on Hugging Face, the website it is downloaded from
WORK_DIR = setup_workdir(RUN_NAME)        # reconnects Google Drive on Colab and returns the run folder
print(WORK_DIR)

## Tools for this notebook
Besides numpy and pandas, this notebook uses **scikit-learn** (imported as `sklearn`), a library of classic machine-learning models. `from module import name` loads one name from a module, so you can write `f1_score(...)` instead of `sklearn.metrics.f1_score(...)`. It is like Java's `import java.util.ArrayList;`.

In [ ]:
# The standard data tools (notebook 00, Part 4).
import numpy as np
import pandas as pd

# The pieces of the baseline model, and the score we use. Each is explained below, where you use it.
from sklearn.pipeline import make_pipeline           # chains several steps into one model
from sklearn.preprocessing import StandardScaler      # puts every input number on the same scale
from sklearn.linear_model import LogisticRegression   # the classifier
from sklearn.metrics import f1_score                  # the macro-F1 score

In [ ]:
df = load_dataset()  # the cleaned table from notebook 01: one row per enzyme, all five species
print(f"{len(df):,} proteins")
df["split"].value_counts()

## From a protein to a list of numbers
A classifier needs numbers as input, not letters. ESM-2 turns a protein's sequence into numbers.

Recall from your transformer presentation: a transformer turns each token of its input into a vector (a list of numbers), and every layer updates those vectors by letting each token pay attention to the others. ESM-2 is a transformer whose tokens are amino acids. After its last layer (it has 6), every amino acid has a list of 320 numbers that describes it *in the context of the whole protein*.

Proteins have different lengths, but a classifier needs the same number of inputs for every protein. So we average: for each of the 320 positions in the list, take the mean over all the amino acids of the protein. This is called **mean pooling**. The two special tokens ESM-2 adds at the start and end of every sequence are left out of the average, and so is the *padding*: filler tokens that make short proteins as long as the others in a batch, so a group of proteins can run through the model together.

The result is one list of 320 numbers per protein, its **embedding**: a summary of the protein that a computer can compare and learn from. A single number in it means nothing on its own; the pattern of all 320 is what carries information.

Here is mean pooling on a made-up protein that is small enough to read.

In [ ]:
# A made-up "protein" of 4 amino acids, each described by 3 numbers (ESM-2 uses 320; 3 keeps it readable).
# np.array of a list of lists makes a 2D array, like Java's double[][]: one row per amino acid.
per_residue = np.array([
    [0.2, 1.0, -0.5],
    [0.4, 0.0, -0.1],
    [0.0, 2.0,  0.3],
    [0.2, 1.0,  0.3],
])
print(per_residue.shape)                  # (4, 3): 4 rows (amino acids) and 3 columns (numbers)
toy_embedding = per_residue.mean(axis=0)  # axis=0 averages down each column, over all the rows
print(toy_embedding)                      # one list of 3 numbers that summarizes the whole protein

## Embeddings from frozen ESM-2
In this notebook ESM-2 is **frozen**: we only use it to read proteins, and none of its 8 million parameters change. The only thing that learns is a small classifier on top. That gives a fair "before" picture. In notebook 03 you fine-tune this same ESM-2 model on the same training proteins, and its classifier reads the same average, so any difference between the two comes from fine-tuning changing ESM-2's own weights.

The next cell computes the embedding of every protein in `df`. The first time, it downloads the model and takes a few minutes; then it saves the embeddings to your Drive. If you run it again later (on another day, or after Colab resets), it finds the saved file and loads it in seconds.

In [ ]:
# cached_embeddings (from our helpers) runs frozen ESM-2 over every sequence in df, mean-pools each one,
# and saves the result in your run folder as embeddings_base.npz.
X = cached_embeddings(df, WORK_DIR / "embeddings_base.npz", MODEL_NAME)
print(X.shape)  # (number of proteins, 320): row i is the embedding of row i of df
print(X[0, :8])  # the first 8 of the 320 numbers for the first protein. X[row, columns] picks from a 2D array.

## Inputs and labels
Machine-learning code usually calls the inputs `X` (capital, because it is a 2D table of numbers) and the labels `y`. Our label is the EC class. Both are lined up with the rows of `df`: row `i` of `X`, item `i` of `y`, and row `i` of `df` are the same protein.

`test_mask` marks the test proteins: the human test set plus every mouse, fly, yeast, and *E. coli* protein. `human_test_mask` marks only the human test proteins, the ones that give the human score.

In [ ]:
y = df["ec_class"].to_numpy()  # the label of every protein, as a numpy array

# Masks (notebook 00, Part 6): True for the rows we want. .to_numpy() turns each into a plain numpy array.
test_mask = (df["split"] == "test").to_numpy()
human_test_mask = ((df["species"] == "human") & (df["split"] == "test")).to_numpy()
print(f"test proteins (all species): {test_mask.sum():,}   human test proteins: {human_test_mask.sum():,}")

**Your turn.** The classifier may learn only from the training proteins. Only human proteins are in the training set (the other species are all test), so `split == "train"` picks the human training proteins. A mask works on a 2D array too: `X[mask]` keeps the rows where the mask is `True`, with all 320 columns.

In [ ]:
# TODO: Make X_train (the embeddings of the training proteins) and y_train (their labels).
# Hint: make train_mask the same way as test_mask, with "train"; then index X and y with it.
X_train = None  # replace None with your code
y_train = None  # replace None with your code

The check cells in this notebook end with a semicolon, `check_...(...);`. The semicolon hides the extra `True` or `False` you saw under each check in notebook 00, so you see only the one line.

In [ ]:
check_train_arrays(X_train, y_train, X, df);

## Class imbalance
Notebook 01 showed that the classes are far from equal. Here are the training labels again. A classifier that learned only to say "2" or "3" would already get most training proteins right while never finding the rare classes, so we will tell it to care about every class.

In [ ]:
# How many training proteins are in each class. pd.Series(...) wraps the array so we can use value_counts.
pd.Series(y_train).value_counts().sort_index()

## Logistic regression
**Logistic regression** is like the simplest possible neural network from your presentation: one layer of weights, with no hidden layers. For each of the 7 classes it learns one weight for each of the 320 embedding numbers, plus one extra number (the bias). To score a protein for a class, it multiplies each embedding number by its weight, adds them all up, and adds the bias. The softmax function turns the 7 scores into 7 probabilities that add up to 1, and the prediction is the class with the highest probability. Training adjusts the weights until the true class gets a high probability for the training proteins.

Our baseline is a short **pipeline**, a chain of steps that acts like one model:
1. `StandardScaler()` rescales each of the 320 numbers so that, over the training proteins, it has mean 0 and standard deviation 1. With all 320 on the same scale, training finds good weights more easily.
2. `LogisticRegression(max_iter=2000, class_weight="balanced")` is the classifier. `max_iter=2000` lets training take up to 2,000 improvement steps (iterations), enough to finish. `class_weight="balanced"` makes a mistake on a rare class cost more than a mistake on a common one, in proportion to how rare the class is, so the model cannot get a good score by ignoring classes 4 to 7.

`make_pipeline(step1, step2)` chains the steps. Calling `.fit(X_train, y_train)` on the pipeline trains every step, and `.predict(X)` runs every step to give a predicted class for each row. `.fit` returns the trained model, so you can build and train it in one line.

In [ ]:
# TODO: Build the baseline (a StandardScaler followed by a LogisticRegression) and train it on X_train and y_train.
#       Store the trained model in clf (short for classifier).
# Hint: make_pipeline with StandardScaler() and the LogisticRegression(...) shown above, then .fit on the training arrays.
clf = None  # replace None with your code

In [ ]:
check_fitted_classifier(clf);

In [ ]:
# Look inside the trained model. clf[-1] is the last step of the pipeline, the logistic regression
# (a negative index counts from the end, as with lists in notebook 00).
# coef_ holds the weights it learned: one row per class, one column per embedding number.
print(clf[-1].coef_.shape)
print(clf[-1].classes_)  # the class of each row of weights
print(f"{clf[-1].coef_.size + clf[-1].intercept_.size:,} numbers learned (weights plus biases); ESM-2 has about 8 million")

## The score: macro-F1
Notebook 01 showed why accuracy alone can fool us: when most proteins are in classes 2 and 3, a model can be right most of the time and still never find a class 6 enzyme. So we use a score that treats every class as equally important.

For one class, say class 6:
- **Precision**: of the proteins the model called class 6, the fraction that really are class 6.
- **Recall**: of the proteins that really are class 6, the fraction the model called class 6.
- **F1** combines the two into one number, `2 * precision * recall / (precision + recall)`. It is high only when both are high.

**Macro-F1** computes F1 for each of the 7 classes and takes their plain average, so a rare class counts as much as a common one. It runs from 0 (useless) to 1 (perfect). A model that never predicts a class gets an F1 of 0 for it, which pulls the average down.

`f1_score(y_true, y_pred, average="macro")` computes it: `y_true` holds the real classes and `y_pred` the predicted ones, in the same order.

**Prediction (write this before you run the next cell):** What macro-F1 do you expect the baseline to get on the human test proteins? Will it do better on the common classes or the rare ones? Why?

In [ ]:
# TODO: Predict the class of each human test protein and store the predictions in y_pred_human.
#       Then score them with macro-F1 against the true classes, and store the score in human_test_f1.
# Hint: clf.predict on the human test rows of X (human_test_mask), then f1_score(true, predicted, average="macro").
y_pred_human = None  # replace None with your code
human_test_f1 = None  # replace None with your code

In [ ]:
check_macro_f1(human_test_f1, y[human_test_mask], y_pred_human);

In [ ]:
print(f"Baseline macro-F1 on human test proteins: {human_test_f1:.3f}")

# The F1 of each class separately: average=None skips the averaging.
# labels=list(CLASS_NAMES) lists the classes 1 to 7 (the keys of the CLASS_NAMES dict) in order.
# zero_division=0 scores a class the model never predicts as 0, instead of printing a warning.
class_f1 = f1_score(y[human_test_mask], y_pred_human, average=None, labels=list(CLASS_NAMES), zero_division=0)
# Show them as a Series, with the class names (the values of the dict) as row labels.
pd.Series(class_f1, index=list(CLASS_NAMES.values()), name="F1").round(2)

**Compare:** Was your prediction right? What surprised you?

## Predictions for every test protein
Notebook 04 scores the models on all five species, so it needs the baseline's predictions for every test protein, not just the human ones. You will put them in a table with one row per test protein.

Two pandas tools help:
- `df.loc[mask, ["col1", "col2"]]` picks rows with a mask and columns by name at the same time. Adding `.copy()` makes a separate table, so changing it later does not touch `df`.
- `table["new_column"] = values` adds a column. `values` must have one item per row, in the same order as the table.

In [ ]:
# TODO: Make a table named preds with one row per test protein (every species) and four columns:
#       accession, species, y_true (the real class), and y_pred_baseline (the baseline's predicted class).
# Hint: df.loc[test_mask, [...]].copy() for the first two columns; then y_true from y and y_pred_baseline
#       from clf.predict, both with test_mask.
preds = None  # replace None with your code

In [ ]:
check_prediction_table(preds, df, "baseline");

## The floor, and saving
Notebook 04 also compares both models with the **floor**: the lazy "model" from notebook 01 that ignores the protein and always guesses the most common training class. It shows the score you get with no learning at all. The next cell adds its predictions as one more column, then saves the table to your Drive.

In [ ]:
# value_counts() counts each class; idxmax() gives the label (here, the class) with the biggest count.
floor_class = pd.Series(y_train).value_counts().idxmax()
preds["y_pred_floor"] = floor_class  # a single value fills the whole column

# Put the columns in the order notebook 04 expects, then save as a tab-separated file.
# index=False leaves out pandas' row labels, which notebook 04 does not need.
preds = preds[["accession", "species", "y_true", "y_pred_floor", "y_pred_baseline"]]
preds.to_csv(WORK_DIR / "predictions_baseline.tsv", sep="\t", index=False)
print(f"Saved {len(preds):,} rows to {WORK_DIR / 'predictions_baseline.tsv'}")
preds.head()

In [ ]:
# The floor and the baseline on the human test proteins, scored two ways.
# preds holds only test proteins, so its human rows are exactly the human test proteins.
human = preds[preds["species"] == "human"]
floor_accuracy = (human["y_true"] == human["y_pred_floor"]).mean()        # the fraction of guesses that are right
baseline_accuracy = (human["y_true"] == human["y_pred_baseline"]).mean()
floor_f1 = f1_score(human["y_true"], human["y_pred_floor"], average="macro", zero_division=0)
print(f"floor:    accuracy {floor_accuracy:.3f}   macro-F1 {floor_f1:.3f}")
print(f"baseline: accuracy {baseline_accuracy:.3f}   macro-F1 {human_test_f1:.3f}")

Look at the floor's two scores. Which one makes the lazy guess look better than it is? That is the reason we report macro-F1.

## In your own words
**Question 1.** What does the logistic regression learn from the embeddings? Describe what its weights are and how it uses them to pick a class for a protein it has never seen.

**Your answer:**

**Question 2.** The baseline uses the same ESM-2 model that you will fine-tune in notebook 03, kept frozen. Why the same model? What would go wrong with the comparison if the baseline used a different model?

**Your answer:**

## Done
Your embeddings and the baseline's predictions are saved on your Drive.

**Check-in 1 comes next.** Your mentor will ask you to explain, in your own words and without AI help: the data and the label, class imbalance, what an embedding is and how mean pooling makes one, and what logistic regression does with the embeddings. Reread this notebook's explanations and your answers first.

After the check-in: notebook 03, where you fine-tune ESM-2.